# Module 1: ROI Detection
# Problem Statement
**Input**: Ảnh đề mẫu (template) được chụp bằng điện thoại

**Output**: File json chứa tọa độ bbox khoanh các vùng điền đáp án của học sinh, xếp theo thứ tự, mapping theo câu hỏi

## Methodology
1. Chuyển sang ảnh nhị phân
2. Tìm các dấu chấm => Gom các dấu chấm cách đều nhau trên một dòng
3. Gom các bbox trên một dòng lại thành một => Gom các hàng cách đều nhau lại thành một bbox lớn
4. Tăng kích thước vùng bao để bắt trọn vẹn chữ học sinh (Tăng giống nhau hết)
5. Tìm các bảng => Bỏ các bảng thuộc đề => Tăng kích thước vùng bao
6. Trong một trang, dựa vào tọa độ trục tung sắp xếp các ROIs theo thứ tự từ trên xuống
7. Ghi vào file json

## Discussion
- Lỗi duy nhất ở trang 5 của mã đề 2, do ảnh chụp mờ => Nhạy cảm với nhiễu hạt
- Ở trang 1, vì có gồm các bảng ghi thông tin sinh viên => Thuật toán sẽ bắt mấy dấu .... trong bảng đó => Nên che đoạn đó lại trước khi đưa vào notebook này (còn thuật toán bắt bảng thì đã né được)
- Nếu che đoạn đó lại thì không biết có ảnh hưởng module align sau này hay không

## Future Work
- Lấy code che thông tin sinh viên của anartt
- Code hiện tại vẫn còn bắt các dấu chấm điền trong ô thông tin sinh viên, nếu sau này có sử dụng cho các bài cần đến mssv thì để yên, không thì che lại từ đầu.
- Nếu muốn thay đổi kích thước vùng bao của ROI, thì bỏ vô phần visualize xem rồi vô file json sửa tay, vì code nó đang tăng size giống nhau hết

In [1]:
# ==========================================
# CELL 1: IMPORTS
# ==========================================
import os
import json
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math

print("Đã load xong thư viện!")

Đã load xong thư viện!


In [2]:
# ==========================================
# CELL 2: TIỀN XỬ LÝ ẢNH
# ==========================================

def preprocess_image(img):
    """
    Hàm tiền xử lý ảnh: Chuyển ảnh màu thành ảnh nhị phân (đen/trắng) 
    để chuẩn bị cho việc tìm contours.
    
    Args:
        img: Ảnh gốc dạng ma trận (đọc bằng cv2.imread)
        
    Returns:
        binary: Ảnh nhị phân (nền đen, mực in màu trắng)
    """
    # Chuyển đổi sang ảnh xám
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 1. LÀM MỜ (BLUR): Xóa nhiễu bề mặt giấy
    # Làm mờ nhẹ giúp các đốm nhiễu nhỏ tan biến, đồng thời làm các dấu chấm nét đứt tròn trịa hơn
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # 2. ADAPTIVE THRESHOLDING: Nhị phân hóa cắt ngưỡng thông minh
    # Tham số C=25 (rất cao) sẽ ép các vùng mờ/nhiễu thành màu Đen (nền), chỉ giữ lại mực in đậm là Trắng
    binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY_INV, 21, 25)
    
    # (Tùy chọn bổ sung từ các lần tối ưu trước: Nếu ảnh cực mờ, có thể bật dòng dưới đây)
    # binary = cv2.medianBlur(binary, 3)
    
    return binary

In [3]:
def extract_dotted_lines(binary):
    """
    Tìm và trích xuất các đoạn nét đứt (cả ngắn và dài) từ ảnh nhị phân.
    
    Args:
        binary: Ảnh nhị phân đã qua tiền xử lý.
        
    Returns:
        valid_short_lines: Danh sách Bbox của các đoạn nét đứt [{'x', 'y', 'w', 'h'}, ...]
        avg_h: Chiều cao trung bình của 1 dấu chấm (dùng làm mốc cho các hàm gộp sau này).
    """
    # --- 1. Tìm và lọc contours lấy "hạt tiêu" ---
    contours, _ = cv2.findContours(binary, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

    dots = []
    avg_h = 0
    if len(contours) > 0:
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            # Bộ lọc kích thước và hình dáng "Tròn trịa"
            if 1 <= h <= 10 and 1 <= w <= 10:
                aspect_ratio = w / float(h)
                if 0.6 <= aspect_ratio <= 1.6: 
                    dots.append({'x': x, 'y': y, 'w': w, 'h': h, 'cx': x + w // 2, 'cy': y + h // 2})

    # Tính chiều cao trung bình của một dấu chấm
    if len(dots) > 0:
        avg_h = sum([d['h'] for d in dots]) / len(dots)
        avg_h = max(3, avg_h) # Giới hạn tối thiểu 3px

    # --- 2. Thuật toán gom nhóm logic: Bắt cặp hạt tiêu ---
    GAP_MULT_MIN = 0.5 
    GAP_MULT_MAX = 2.5
    Y_DIFF_TOLERANCE = 1.0 

    valid_short_lines = []

    if len(dots) > 1:
        # Sắp xếp chấm theo toạ độ Y tăng dần
        dots_y_sorted = sorted(dots, key=lambda d: d['cy'])
        
        current_y_line = dots_y_sorted[0]['cy']
        row_dots = [dots_y_sorted[0]]
        
        # Gom chấm thô theo dòng
        raw_lines = []
        for i in range(1, len(dots_y_sorted)):
            d = dots_y_sorted[i]
            if abs(d['cy'] - current_y_line) <= (avg_h * Y_DIFF_TOLERANCE):
                row_dots.append(d)
            else:
                if len(row_dots) >= 2: 
                    raw_lines.append(sorted(row_dots, key=lambda d: d['x']))
                row_dots = [d]
                current_y_line = d['cy']
        if len(row_dots) >= 2:
            raw_lines.append(sorted(row_dots, key=lambda d: d['x']))

        # --- 3. Kiểm tra độ đều đặn của từng cặp chấm trong dòng ---
        for line in raw_lines:
            current_chain = [line[0]]
            for i in range(1, len(line)):
                d1 = current_chain[-1]
                d2 = line[i]
                
                gap_x = d2['x'] - (d1['x'] + d1['w'])
                
                if (avg_h * GAP_MULT_MIN) <= gap_x <= (avg_h * GAP_MULT_MAX):
                    current_chain.append(d2)
                else:
                    # Ngắt chuỗi, chốt Bbox
                    if len(current_chain) >= 4:
                        min_x = current_chain[0]['x']
                        max_x = current_chain[-1]['x'] + current_chain[-1]['w']
                        min_y = min([d['y'] for d in current_chain])
                        max_y = max([d['y'] + d['h'] for d in current_chain])
                        
                        valid_short_lines.append({'x': min_x, 'y': min_y, 'w': max_x - min_x, 'h': max_y - min_y})
                    current_chain = [d2]
            
            # Chốt sổ chuỗi cuối của dòng
            if len(current_chain) >= 4:
                min_x = current_chain[0]['x']
                max_x = current_chain[-1]['x'] + current_chain[-1]['w']
                min_y = min([d['y'] for d in current_chain])
                max_y = max([d['y'] + d['h'] for d in current_chain])
                valid_short_lines.append({'x': min_x, 'y': min_y, 'w': max_x - min_x, 'h': max_y - min_y})

    return valid_short_lines, avg_h

In [4]:
def group_dotted_bboxes(valid_short_lines, avg_h):
    """
    Gom các đoạn nét đứt rời rạc thành các block (đoạn văn/vùng điền đáp án lớn).
    Hỗ trợ xử lý trường hợp dòng đầu tiên bị ngắn do vướng câu hỏi.
    
    Args:
        valid_short_lines: Danh sách các bbox đoạn nét đứt.
        avg_h: Chiều cao trung bình của 1 dấu chấm (dùng làm mốc khoảng cách).
        
    Returns:
        final_bboxes: Danh sách các bbox lớn sau khi đã gom nhóm chiều ngang và dọc.
    """
    final_bboxes = []
    
    # --- 1. GIAI ĐOẠN 1: GOM CÁC BBOX THEO CHIỀU NGANG ---
    horizontal_merged_lines = []

    if len(valid_short_lines) > 0:
        sorted_by_y = sorted(valid_short_lines, key=lambda b: (b['y'], b['x']))
        current_hline = sorted_by_y[0]
        
        for i in range(1, len(sorted_by_y)):
            next_box = sorted_by_y[i]
            if abs(next_box['y'] - current_hline['y']) <= (avg_h * 2):
                min_x = min(current_hline['x'], next_box['x'])
                min_y = min(current_hline['y'], next_box['y'])
                max_x = max(current_hline['x'] + current_hline['w'], next_box['x'] + next_box['w'])
                max_y = max(current_hline['y'] + current_hline['h'], next_box['y'] + next_box['h'])
                current_hline = {'x': min_x, 'y': min_y, 'w': max_x - min_x, 'h': max_y - min_y}
            else:
                horizontal_merged_lines.append(current_hline)
                current_hline = next_box
        horizontal_merged_lines.append(current_hline)

    # --- 2. GIAI ĐOẠN 2: GOM CÁC BBOX THEO CHIỀU DỌC (TẠO BLOCK) ---
    final_blocks = []

    if len(horizontal_merged_lines) > 0:
        horizontal_merged_lines = sorted(horizontal_merged_lines, key=lambda b: b['y'])
        current_block = [horizontal_merged_lines[0]]
        expected_gap = None
        
        for i in range(1, len(horizontal_merged_lines)):
            prev_box = current_block[-1]
            next_box = horizontal_merged_lines[i]
            
            # Tính toán các chỉ số so sánh dọc và ngang
            gap_y = next_box['y'] - (prev_box['y'] + prev_box['h'])
            width_ratio = min(prev_box['w'], next_box['w']) / max(prev_box['w'], next_box['w'])
            
            # Đoạn giao nhau theo phương ngang (X)
            overlap_x = max(0, min(prev_box['x'] + prev_box['w'], next_box['x'] + next_box['w']) - max(prev_box['x'], next_box['x']))
            
            # Tính tỷ lệ giao nhau so với chiều dài của dòng NGẮN HƠN
            min_w = min(prev_box['w'], next_box['w'])
            overlap_ratio = overlap_x / min_w if min_w > 0 else 0
            
            # Kiểm tra lề phải
            right_diff = abs((prev_box['x'] + prev_box['w']) - (next_box['x'] + next_box['w']))
            is_right_aligned = right_diff <= (avg_h * 5)
            
            # Điều kiện gộp trục X
            is_x_valid = (width_ratio >= 0.7) or (overlap_ratio >= 0.8) or is_right_aligned
            
            # Điều kiện khoảng cách dòng không đổi
            is_gap_valid = False
            if gap_y > 0 and gap_y < (avg_h * 30):
                if expected_gap is None:
                    is_gap_valid = True 
                else:
                    if abs(gap_y - expected_gap) <= (avg_h * 5): 
                        is_gap_valid = True

            # Ra quyết định
            if is_x_valid and is_gap_valid:
                current_block.append(next_box)
                if len(current_block) >= 2:
                    gaps = [current_block[k]['y'] - (current_block[k-1]['y'] + current_block[k-1]['h']) for k in range(1, len(current_block))]
                    expected_gap = sum(gaps) / len(gaps)
            else:
                final_blocks.append(current_block)
                current_block = [next_box]
                expected_gap = None
                
        final_blocks.append(current_block)

    # --- 3. TẠO BBOX TO CHO TỪNG BLOCK ---
    for block in final_blocks:
        min_x = min([b['x'] for b in block])
        min_y = min([b['y'] for b in block])
        max_x = max([b['x'] + b['w'] for b in block])
        max_y = max([b['y'] + b['h'] for b in block])
        
        final_bboxes.append({'x': min_x, 'y': min_y, 'w': max_x - min_x, 'h': max_y - min_y})

    return final_bboxes

In [5]:
def pad_dotted_bboxes(final_bboxes, img_height, img_width):
    """
    Mở rộng kích thước (padding) cho các bbox nét đứt để bao trọn vùng không gian rộng hơn.
    Đồng thời giới hạn tọa độ không bị tràn ra ngoài kích thước ảnh.
    
    Args:
        final_bboxes: Danh sách các bbox nét đứt sau khi gom nhóm.
        img_height: Chiều cao ảnh gốc.
        img_width: Chiều rộng ảnh gốc.
        
    Returns:
        expanded_bboxes: Danh sách bbox sau khi đã mở rộng (có kèm nhãn 'type').
    """
    expanded_bboxes = []

    for box in final_bboxes:
        x, y, w, h = box['x'], box['y'], box['w'], box['h']
        
        # --- 1. Tính toán tọa độ mở rộng ---
        y_start = y - 60
        y_end = y + h + 20
        
        x_start = x - 50
        x_end = x + w + 100
        
        # --- 2. Cắt xén (clip) tọa độ ---
        # Đảm bảo tọa độ không bao giờ nhỏ hơn 0 hoặc lớn hơn kích thước ảnh gốc
        x_start = max(0, x_start)
        y_start = max(0, y_start)
        x_end = min(img_width, x_end)
        y_end = min(img_height, y_end)
        
        # --- 3. Cập nhật lại w, h mới ---
        new_w = x_end - x_start
        new_h = y_end - y_start
        
        expanded_bboxes.append({
            'x': x_start, 
            'y': y_start, 
            'w': new_w, 
            'h': new_h,
            'type': 'fill_in_blank' # Gắn nhãn để phân biệt với Bảng biểu
        })

    return expanded_bboxes

In [6]:
def extract_tables(img):
    """
    Nhận diện và trích xuất khung bảng biểu bằng phương pháp hình thái học (Morphology).
    
    Args:
        img: Ảnh gốc dạng ma trận (đọc bằng cv2.imread).
        
    Returns:
        table_bboxes: Danh sách tọa độ thô của các bảng [{'x', 'y', 'w', 'h'}, ...].
        binary_table: Ảnh nhị phân dùng riêng cho xử lý bảng (chứa cả viền và chữ).
        table_mask: Mặt nạ (mask) chỉ chứa khung lưới của các bảng (không có chữ).
    """
    # --- 1. Tiền xử lý ảnh chuyên biệt cho nhận diện đường thẳng ---
    gray_table = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    binary_table = cv2.adaptiveThreshold(gray_table, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                         cv2.THRESH_BINARY_INV, 15, 5)

    img_height, img_width = binary_table.shape

    # --- 2. Khởi tạo các Kernel (Bộ lọc) ---
    scale = 40 
    horizontal_size = max(20, img_width // scale) 
    vertical_size = max(20, img_height // scale)

    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (horizontal_size, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, vertical_size))

    # --- 3. Trích xuất đường thẳng ---
    horizontal_lines = cv2.morphologyEx(binary_table, cv2.MORPH_OPEN, horizontal_kernel)
    vertical_lines = cv2.morphologyEx(binary_table, cv2.MORPH_OPEN, vertical_kernel)

    # --- 4. Kết hợp tạo bộ khung lưới (Grid) ---
    table_mask = cv2.add(horizontal_lines, vertical_lines)

    kernel_dilate = np.ones((3,3), np.uint8)
    table_mask = cv2.dilate(table_mask, kernel_dilate, iterations=1)

    # --- 5. Tìm Contours và lọc ra Bảng ---
    contours_table, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    table_bboxes = []
    min_table_area = (img_width * img_height) * 0.01 

    for cnt in contours_table:
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h
        
        # Điều kiện lọc rác
        if w > 100 and h > 50 and area > min_table_area:
            table_bboxes.append({'x': x, 'y': y, 'w': w, 'h': h})

    return table_bboxes, binary_table, table_mask

In [7]:
def filter_and_pad_tables(table_bboxes, binary_table, table_mask, img_height, img_width, text_density_threshold=2.0):
    """
    Lọc bỏ các bảng chứa nội dung đề bài (nhiều chữ) và mở rộng (padding) 
    tọa độ cho các bảng để học sinh điền đáp án (ít/không có chữ).
    
    Args:
        table_bboxes: Danh sách tọa độ thô của các bảng.
        binary_table: Ảnh nhị phân chứa cả viền bảng và chữ.
        table_mask: Mask chỉ chứa khung lưới bảng.
        img_height: Chiều cao ảnh gốc.
        img_width: Chiều rộng ảnh gốc.
        text_density_threshold: Ngưỡng % mật độ chữ để phân loại (mặc định 1.5%).
        
    Returns:
        expanded_answer_table_bboxes: Danh sách bbox bảng đáp án đã được padding.
    """
    expanded_answer_table_bboxes = []
    ignored_question_tables = []
    
    for box in table_bboxes:
        x, y, w, h = box['x'], box['y'], box['w'], box['h']
        
        # --- BƯỚC 1: KIỂM TRA MẬT ĐỘ CHỮ ĐỂ LỌC BẢNG ---
        roi_binary = binary_table[y:y+h, x:x+w]
        roi_grid = table_mask[y:y+h, x:x+w]
        roi_text_only = cv2.subtract(roi_binary, roi_grid)
        text_pixels = cv2.countNonZero(roi_text_only)
        
        text_density = (text_pixels / (w * h)) * 100
        
        if text_density < text_density_threshold:
            # ĐÂY LÀ BẢNG HỌC SINH -> TIẾN HÀNH MỞ RỘNG (PADDING)
            
            # --- BƯỚC 2: Tính toán tọa độ mở rộng ---
            y_start = y
            y_end = y + h + 50
            
            x_start = x - 50
            x_end = x + w + 50
            
            # --- BƯỚC 3: Cắt xén (clip) tọa độ giới hạn trong khung ảnh ---
            x_start = max(0, x_start)
            y_start = max(0, y_start)
            x_end = min(img_width, x_end)
            y_end = min(img_height, y_end)
            
            # --- BƯỚC 4: Cập nhật kích thước mới ---
            new_w = x_end - x_start
            new_h = y_end - y_start
            
            expanded_answer_table_bboxes.append({
                'x': x_start, 
                'y': y_start, 
                'w': new_w, 
                'h': new_h,
                'type': 'table' # Gắn nhãn luôn để tiện cho bước Mapping sau này
            })
        else:
            # ĐÂY LÀ BẢNG ĐỀ BÀI -> BỎ QUA KHÔNG PADDING
            ignored_question_tables.append(box)

    return expanded_answer_table_bboxes

In [8]:
def map_and_sort_rois(expanded_bboxes, expanded_answer_table_bboxes, y_tolerance=30):
    """
    Gom chung tất cả các loại ROI (nét đứt, bảng biểu) vào một danh sách, 
    sau đó sắp xếp thứ tự không gian từ trên xuống dưới, từ trái qua phải.
    
    Args:
        expanded_bboxes: Danh sách ROI nét đứt đã padding.
        expanded_answer_table_bboxes: Danh sách ROI bảng đáp án đã padding.
        y_tolerance: Sai số Y (pixel) để coi 2 khối nằm trên cùng 1 hàng.
        
    Returns:
        sorted_rois: Danh sách tất cả các ROI đã được sắp xếp đúng thứ tự.
    """
    all_rois = []

    # 1. Đưa các Bbox nét đứt vào danh sách chung
    for box in expanded_bboxes:
        all_rois.append({
            'x': box['x'], 
            'y': box['y'], 
            'w': box['w'], 
            'h': box['h'],
            'type': box.get('type', 'fill_in_blank'), 
            'center_y': box['y'] + box['h'] // 2
        })

    # 2. Đưa các Bbox bảng biểu vào danh sách chung
    for box in expanded_answer_table_bboxes:
        all_rois.append({
            'x': box['x'], 
            'y': box['y'], 
            'w': box['w'], 
            'h': box['h'],
            'type': box.get('type', 'table'),
            'center_y': box['y'] + box['h'] // 2
        })

    # 3. Thuật toán sắp xếp không gian (Top-to-Bottom, Left-to-Right)
    sorted_rois = []

    if len(all_rois) > 0:
        # Sắp xếp thô theo trục Y trước
        all_rois.sort(key=lambda b: b['center_y'])
        
        current_row = [all_rois[0]]
        for i in range(1, len(all_rois)):
            box = all_rois[i]
            # Nếu box hiện tại nằm cùng hàng với box trước đó
            if abs(box['center_y'] - current_row[-1]['center_y']) <= y_tolerance:
                current_row.append(box)
            else:
                # Chốt hàng hiện tại, sắp xếp theo trục X và đẩy vào mảng đích
                sorted_rois.extend(sorted(current_row, key=lambda b: b['x']))
                current_row = [box]
                
        # Chốt hàng cuối cùng
        sorted_rois.extend(sorted(current_row, key=lambda b: b['x']))

    return sorted_rois

# Hàm chạy chính

In [9]:
def process_folder(input_folder, output_json_path):
    final_results = {}
    image_paths = glob.glob(os.path.join(input_folder, '*.[jp][pn]*[g]'))
    
    print(f"📁 Tìm thấy {len(image_paths)} ảnh trong thư mục: {input_folder}")
    
    for img_path in image_paths:
        filename = os.path.basename(img_path)
        print(f"\n⏳ Đang xử lý: {filename}...")
        
        # 0. Đọc ảnh
        original_img = cv2.imread(img_path)
        if original_img is None:
            print(f"⚠️ Lỗi: Không thể đọc ảnh {filename}")
            continue
            
        img_height, img_width = original_img.shape[:2]
        
        # --- QUY TRÌNH NÉT ĐỨT ---
        # 1. Tiền xử lý
        binary_clean = preprocess_image(original_img)
        # 2. Trích xuất nét đứt thô
        valid_short_lines, avg_h = extract_dotted_lines(binary_clean)
        # 3. Gom nhóm nét đứt
        final_bboxes = group_dotted_bboxes(valid_short_lines, avg_h)
        # 4. Mở rộng (padding)
        expanded_bboxes = pad_dotted_bboxes(final_bboxes, img_height, img_width)
        
        # --- QUY TRÌNH BẢNG BIỂU ---
        # 5. Trích xuất lưới bảng
        table_bboxes, binary_table, table_mask = extract_tables(original_img)
        # 6. Lọc bảng đáp án và Padding
        expanded_answer_table_bboxes = filter_and_pad_tables(
            table_bboxes, binary_table, table_mask, img_height, img_width
        )
        
        # --- GOM NHÓM & SẮP XẾP ---
        # 7. Hợp nhất thành danh sách cuối cùng
        sorted_rois = map_and_sort_rois(expanded_bboxes, expanded_answer_table_bboxes)
        
        # --- XUẤT RA JSON ---
        formatted_rois = []
        for roi in sorted_rois:
            # Format JSON bạn yêu cầu: [x_min, y_min, x_max, y_max]
            x_min = roi['x']
            y_min = roi['y']
            x_max = roi['x'] + roi['w']
            y_max = roi['y'] + roi['h']
            
            formatted_rois.append([x_min, y_min, x_max, y_max])
            
        final_results[filename] = {
            "total_blocks": len(formatted_rois),
            "rois": formatted_rois
        }
        
        print(f"✅ Hoàn tất {filename}: Tìm thấy {len(formatted_rois)} blocks.")

    # Ghi file JSON
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_results, f, ensure_ascii=False, indent=4)
        
    print(f"\n🚀 TẤT CẢ ĐÃ HOÀN THÀNH! Kết quả lưu tại: {output_json_path}")

In [10]:
# ==========================================
# CÁCH CHẠY THỰC TẾ
# ==========================================
input_folder = "/kaggle/input/datasets/anartt/dts-hki2526/output/Made_1/Template_1"  # Thay bằng tên thư mục chứa ảnh của bạn
output_json_path = "Template_1_ROIs.json"
process_folder(input_folder, output_json_path)

📁 Tìm thấy 9 ảnh trong thư mục: /kaggle/input/datasets/anartt/dts-hki2526/output/Made_1/Template_1

⏳ Đang xử lý: M  1 - Bn clean cha lm_5.jpg...
✅ Hoàn tất M  1 - Bn clean cha lm_5.jpg: Tìm thấy 2 blocks.

⏳ Đang xử lý: M  1 - Bn clean cha lm_4.jpg...
✅ Hoàn tất M  1 - Bn clean cha lm_4.jpg: Tìm thấy 3 blocks.

⏳ Đang xử lý: M  1 - Bn clean cha lm_9.jpg...
✅ Hoàn tất M  1 - Bn clean cha lm_9.jpg: Tìm thấy 3 blocks.

⏳ Đang xử lý: M  1 - Bn clean cha lm_2.jpg...
✅ Hoàn tất M  1 - Bn clean cha lm_2.jpg: Tìm thấy 3 blocks.

⏳ Đang xử lý: M  1 - Bn clean cha lm_3.jpg...
✅ Hoàn tất M  1 - Bn clean cha lm_3.jpg: Tìm thấy 2 blocks.

⏳ Đang xử lý: M  1 - Bn clean cha lm_6.jpg...
✅ Hoàn tất M  1 - Bn clean cha lm_6.jpg: Tìm thấy 2 blocks.

⏳ Đang xử lý: M  1 - Bn clean cha lm_1.jpg...
✅ Hoàn tất M  1 - Bn clean cha lm_1.jpg: Tìm thấy 3 blocks.

⏳ Đang xử lý: M  1 - Bn clean cha lm_8.jpg...
✅ Hoàn tất M  1 - Bn clean cha lm_8.jpg: Tìm thấy 2 blocks.

⏳ Đang xử lý: M  1 - Bn clean cha lm_7.jpg..

In [11]:
# ==========================================
# CÁCH CHẠY THỰC TẾ
# ==========================================
input_folder = "/kaggle/input/datasets/anartt/dts-hki2526/output/Made_2/Template_2"  # Thay bằng tên thư mục chứa ảnh của bạn
output_json_path = "Template_2_ROIs.json"
process_folder(input_folder, output_json_path)

📁 Tìm thấy 9 ảnh trong thư mục: /kaggle/input/datasets/anartt/dts-hki2526/output/Made_2/Template_2

⏳ Đang xử lý: M  2 - Bn clean cha lm_8.jpg...
✅ Hoàn tất M  2 - Bn clean cha lm_8.jpg: Tìm thấy 2 blocks.

⏳ Đang xử lý: M  2 - Bn clean cha lm_9.jpg...
✅ Hoàn tất M  2 - Bn clean cha lm_9.jpg: Tìm thấy 3 blocks.

⏳ Đang xử lý: M  2 - Bn clean cha lm_1.jpg...
✅ Hoàn tất M  2 - Bn clean cha lm_1.jpg: Tìm thấy 4 blocks.

⏳ Đang xử lý: M  2 - Bn clean cha lm_6.jpg...
✅ Hoàn tất M  2 - Bn clean cha lm_6.jpg: Tìm thấy 2 blocks.

⏳ Đang xử lý: M  2 - Bn clean cha lm_4.jpg...
✅ Hoàn tất M  2 - Bn clean cha lm_4.jpg: Tìm thấy 3 blocks.

⏳ Đang xử lý: M  2 - Bn clean cha lm_5.jpg...
✅ Hoàn tất M  2 - Bn clean cha lm_5.jpg: Tìm thấy 4 blocks.

⏳ Đang xử lý: M  2 - Bn clean cha lm_7.jpg...
✅ Hoàn tất M  2 - Bn clean cha lm_7.jpg: Tìm thấy 3 blocks.

⏳ Đang xử lý: M  2 - Bn clean cha lm_3.jpg...
✅ Hoàn tất M  2 - Bn clean cha lm_3.jpg: Tìm thấy 2 blocks.

⏳ Đang xử lý: M  2 - Bn clean cha lm_2.jpg..

# Visualize các ROI đã detect được
Phần này để quan sát và debug, có hiển thị grid, dễ dàng điều chỉnh tọa độ theo ý muốn.

In [12]:
# # ==========================================
# # CELL VISUALIZATION: ĐỌC JSON VÀ VẼ BBOX LÊN TỪNG TRANG
# # ==========================================
# import os
# import json
# import cv2
# import matplotlib.pyplot as plt

# def visualize_json_results(input_folder, json_path):
#     """
#     Đọc file JSON đã xuất và vẽ bounding box lên từng ảnh gốc trong thư mục 
#     để kiểm tra xem chương trình bắt ROI có chuẩn xác không.
#     """
#     # 1. Đọc file JSON chứa tọa độ
#     if not os.path.exists(json_path):
#         print(f"⚠️ Không tìm thấy file JSON tại: {json_path}")
#         return
        
#     with open(json_path, 'r', encoding='utf-8') as f:
#         results = json.load(f)
        
#     print(f"🔍 Đang trực quan hóa kết quả cho {len(results)} trang template...\n")
    
#     # 2. Lặp qua từng file ảnh đã được lưu trong JSON
#     for filename, data in results.items():
#         img_path = os.path.join(input_folder, filename)
        
#         if not os.path.exists(img_path):
#             print(f"⚠️ Không tìm thấy ảnh: {img_path}")
#             continue
            
#         # Đọc ảnh gốc
#         img = cv2.imread(img_path)
#         rois = data.get("rois", [])
        
#         # 3. Vẽ toàn bộ Bbox lên ảnh gốc
#         for index, roi in enumerate(rois):
#             # Giải nén tọa độ theo đúng format đã lưu trong JSON: [x_min, y_min, x_max, y_max]
#             x_min, y_min, x_max, y_max = roi
            
#             # Vẽ bounding box (Màu xanh lá, viền dày 4px)
#             cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 4)
            
#             # --- Tạo nhãn nền đen chữ xanh để nổi bật trên nền giấy trắng/chữ đen ---
#             text = f"ROI {index + 1}"
#             font_scale = 1.2
#             thickness = 3
            
#             # Đo kích thước chữ để vẽ nền
#             (text_width, text_height), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
            
#             # Vẽ HCN nền đen
#             cv2.rectangle(img, (x_min, y_min - text_height - 15), (x_min + text_width + 10, y_min), (0, 0, 0), -1)
            
#             # Ghi text số thứ tự
#             cv2.putText(img, text, (x_min + 5, y_min - 10), 
#                         cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 255, 0), thickness, cv2.LINE_AA)

#         # 4. Hiển thị full trang ảnh
#         plt.figure(figsize=(18, 24)) # Kích thước hiển thị siêu lớn để nhìn rõ như trang A4
#         plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
#         plt.title(f"Trang: {filename} | Tổng số ROI bắt được: {len(rois)}", fontsize=20, fontweight='bold', color='blue')
        
#         # --- THAY ĐỔI Ở ĐÂY ĐỂ HIỂN THỊ GRID VÀ TRỤC X, Y ---
#         plt.axis('on')  # Bật hiển thị trục tọa độ (thay vì 'off')
#         plt.xlabel("Trục X (pixels)", fontsize=14, fontweight='bold')
#         plt.ylabel("Trục Y (pixels)", fontsize=14, fontweight='bold')
        
#         # Cấu hình grid: màu xám, nét đứt, độ mờ (alpha) vừa phải để không đè mất chữ chữ trên ảnh
#         plt.grid(True, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
        
#         # Tự động điều chỉnh khoảng cách các số trên trục để không bị quá dày đặc
#         plt.locator_params(axis='both', nbins=20) 
#         # ---------------------------------------------------
        
#         plt.show()

In [13]:
# json_path= '/kaggle/input/datasets/camtran3506/temp2test/Template_2_ROIs.json'
# visualize_json_results(input_folder, json_path)